# Retreiver Demonstration

In [3]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader

In [2]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [11]:
# Loading Queries:
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

### Example of a Query:

In [28]:
i = 5

ex_q = [(q_id, queries[q_id]) for q_id in queries][i]
print(ex_q)

('312651', 'how much does an average person make for tutoring')


### Linking Query to Passage

In [33]:
passage_id = list(qrels[ex_q[0]].keys())
print(passage_id)

['616']


### Gathering the Passage

In [38]:
print(corpus[passage_id[0]]['text'])

In-home tutors can earn anywhere from $10 to $80 an hour, depending on the type of lesson, the studentâs skill and age level and the tutorâs experience. Tutors often charge more for older students or those who require more advanced lessons.


# Loading the vector DB, index and json

In [40]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage.index")

# Load Metadata
metadata = []
with open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

### Loading the Trained Query Encoder

In [42]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/RAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ) #.to("cuda")

    outputs = query_encoder(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0] 

    embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)

### Embedding the Quesetion

In [47]:
question = ex_q[1]
print(question)

# Embedding Query
q_emb = encode_query([question]).detach().cpu().numpy()
print('Embedding shape: ', q_emb.shape)

how much does an average person make for tutoring
Embedding shape:  (1, 768)


In [ ]:
# Matching for the top K=10 highest scores
K = 10
scores, ids = index.search(q_emb, K)
candidates = [metadata[i]["text"] for i in ids[0]]
test_df = pd.DataFrame({'scores': scores, 'passages':candidates})
test_df.head(10)

# Generation